# One incident, function by function

This notebook follows **one incident (a bad deploy)** through the same functions the agents call, one cell per step, so you can see exactly what goes in and what comes out at each hop. Nothing here is mocked logic: the connectors, reasoner and action runner are the real classes from `backend/src/triage`. Only the *data source* is a stand-in: a snapshot of real telemetry captured from the demo shop, so it runs offline with no Azure and no API key.

```
Azure Monitor alert
   -> Triage supervisor   (plan)
   -> Metrics agent       (connector -> Evidence -> Finding)      \
      Logs agent          (same)                                   > run in parallel in the real graph
      Change agent        (same)                                  /
   -> Diagnosis agent     (weigh findings; unsure -> asks for more)
   -> Logs agent, second pass -> Diagnosis agent again
   -> Remediation agent   (diagnosis -> one fix from a fixed catalog)
   -> Approval gate       (a person decides)
   -> Action executor     -> Action connector -> the resource (Azure, or the demo shop here)
   -> Postmortem agent    (always runs) -> work item
```

**Parts that were not in your list but are in the real flow:** the *Logs* and *Change* agents (Diagnosis needs all three), the *second Logs pass* (that is the loop), and the *Postmortem agent* that closes every run.

**To try another incident**, change `SCENARIO` in the first code cell to `"memory-leak"` or `"db-pool"` and run all cells again. The explanations below are written for the bad deploy.

**To run it:** needs Python 3.12 and `pip install -e "backend[dev]"`. Or without installing anything, from the repo root:
```
docker run --rm -p 8888:8888 -v "$PWD:/work" -w /work python:3.12-slim sh -c 'pip install -q -e "backend[dev]" jupyterlab && jupyter lab --ip=0.0.0.0 --allow-root --no-browser'
```

> **Cell reference: Setup**
>
> | | |
> |---|---|
> | **Function** | `show(title, obj)` (defined here); `shop_client(name)` (imported from `backend/tests/envhelpers.py`) |
> | **Purpose** | Makes the real backend code importable, picks the incident, and defines the print helper. |
> | **Inputs** | `SCENARIO` (`bad-deploy`, `memory-leak` or `db-pool`) and the folder the notebook was opened in. |
> | **Outputs** | Prints the scenario name. Leaves `SCENARIO`, `show` and `shop_client` for later cells. |

In [1]:
import json
import sys
from pathlib import Path

SCENARIO = "bad-deploy"  # or "memory-leak" or "db-pool"

# Find the backend folder from wherever the notebook was opened.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())
sys.path[:0] = [str(root / "backend" / "src"), str(root / "backend" / "tests")]

from envhelpers import shop_client  # a stand-in "shop" that serves telemetry captured from the real one

def show(title, obj):
    """Print a model or dict as indented JSON under a heading."""
    data = obj.model_dump() if hasattr(obj, "model_dump") else obj
    print(f"--- {title}")
    print(json.dumps(data, indent=2, default=str))

print("scenario:", SCENARIO)

scenario: bad-deploy


## 0. All the inputs the flow can start with

An incident enters as an **Alert**. In production it arrives as Azure Monitor's *common alert schema* JSON posted to `POST /alerts`, which is converted to an `Alert`. Below: the raw payload, every field an `Alert` can have, and the settings that change how the rest behaves.

> **Cell reference: Alert: raw payload and fields**
>
> | | |
> |---|---|
> | **Function** | `Alert.model_fields` (pydantic model in `triage/models.py`) |
> | **Purpose** | Shows what Azure Monitor posts and every field an `Alert` can hold. |
> | **Inputs** | `AZURE_PAYLOAD`, the common-alert-schema JSON written by hand in this cell. |
> | **Outputs** | Prints the payload and one line per `Alert` field (type and default). No state is changed. |

In [2]:
# (a) What Azure Monitor posts to /alerts when an alert rule fires
AZURE_PAYLOAD = {
    "schemaId": "azureMonitorCommonAlertSchema",
    "data": {
        "essentials": {
            "alertId": "/subscriptions/sub-1/providers/Microsoft.AlertsManagement/alerts/b9569717-bc97-4f6b-9ddd-b42d0b8ba9d8",
            "alertRule": "shop-api failed requests above 5%",
            "severity": "Sev1",                       # Sev0..Sev4 -> critical / high / medium / low / low
            "signalType": "Log",
            "monitorCondition": "Fired",              # "Resolved" notifications are ignored
            "monitoringService": "Log Alerts V2",
            "alertTargetIDs": ["/subscriptions/sub-1/resourcegroups/rg-shop/providers/microsoft.web/sites/shop-api"],
            "firedDateTime": "2026-10-15T14:02:11.123Z",
        }
    },
}
show("raw Azure Monitor payload (abridged to what is used)", AZURE_PAYLOAD["data"]["essentials"])

# (b) Every field an Alert can have
from triage.models import Alert
for name, field in Alert.model_fields.items():
    print(f"Alert.{name:<9} type={field.annotation!s:<55} default={field.default if field.default is not None else '(required)'}")
print("severity choices: low, medium, high, critical")

--- raw Azure Monitor payload (abridged to what is used)
{
  "alertId": "/subscriptions/sub-1/providers/Microsoft.AlertsManagement/alerts/b9569717-bc97-4f6b-9ddd-b42d0b8ba9d8",
  "alertRule": "shop-api failed requests above 5%",
  "severity": "Sev1",
  "signalType": "Log",
  "monitorCondition": "Fired",
  "monitoringService": "Log Alerts V2",
  "alertTargetIDs": [
    "/subscriptions/sub-1/resourcegroups/rg-shop/providers/microsoft.web/sites/shop-api"
  ],
  "firedDateTime": "2026-10-15T14:02:11.123Z"
}
Alert.id        type=<class 'str'>                                           default=alert-1
Alert.title     type=<class 'str'>                                           default=PydanticUndefined
Alert.resource  type=<class 'str'>                                           default=PydanticUndefined
Alert.severity  type=typing.Literal['low', 'medium', 'high', 'critical']     default=high
Alert.labels    type=dict[str, str]                                          default=PydanticUndefined

> **Cell reference: Alert: payload to Alert**
>
> | | |
> |---|---|
> | **Function** | `from_common_alert_schema(payload)` (`triage/azure_alerts.py`) |
> | **Purpose** | Turns Azure Monitor's JSON into the `Alert` every agent works from. Also shows the two cases that start no run. |
> | **Inputs** | A payload dict. Needs `schemaId` and `data.essentials` (`alertId`, `alertRule`, `severity`, `monitorCondition`, `alertTargetIDs`). |
> | **Outputs** | `Alert`, or `None` for a resolved notification, or `ValueError` for any other JSON. Leaves `alert` for every later cell. |

In [3]:
# (c) The payload becomes an Alert
from triage.azure_alerts import from_common_alert_schema

alert = from_common_alert_schema(AZURE_PAYLOAD)
show("Alert  <- input to every agent", alert)

# What else could arrive:
resolved = {**AZURE_PAYLOAD, "data": {"essentials": {**AZURE_PAYLOAD["data"]["essentials"], "monitorCondition": "Resolved"}}}
print("\nresolved notification  ->", from_common_alert_schema(resolved), "(no run is started)")
try:
    from_common_alert_schema({"schemaId": "other"})
except ValueError as e:
    print("wrong schema           -> ValueError:", e)

--- Alert  <- input to every agent
{
  "id": "b9569717-bc97-4f6b-9ddd-b42d0b8ba9d8",
  "title": "shop-api failed requests above 5%",
  "resource": "shop-api",
  "severity": "high",
  "labels": {
    "source": "azure-monitor",
    "rule": "shop-api failed requests above 5%",
    "signal_type": "Log",
    "monitoring_service": "Log Alerts V2",
    "fired_at": "2026-10-15T14:02:11.123Z",
    "resource_id": "/subscriptions/sub-1/resourcegroups/rg-shop/providers/microsoft.web/sites/shop-api"
  }
}

resolved notification  -> None (no run is started)
wrong schema           -> ValueError: Expected schemaId 'azureMonitorCommonAlertSchema'. Enable the common alert schema on the Action Group's webhook.


> **Cell reference: Settings that change the flow**
>
> | | |
> |---|---|
> | **Function** | None (a table of environment settings) |
> | **Purpose** | Lists what is configured outside the alert: mode, reasoner, token, wait time, round cap, Azure variables. |
> | **Inputs** | Nothing is read. The table is written by hand. |
> | **Outputs** | Prints the table. |

In [4]:
# (d) Settings that change the flow (read from the environment, not from the alert)
SETTINGS = {
    "TRIAGE_MODE":        "demo | env | azure: which connectors the agents use (here: the captured shop snapshot, as in env)",
    "TRIAGE_REASONER":    "rules (default) | openai: who does the judgment (here: rules)",
    "TRIAGE_ALERT_TOKEN": "optional secret the /alerts webhook must send as ?token=",
    "TRIAGE_VERIFY_WAIT": "seconds the executor waits before checking the fix worked (here 0, nothing live to wait for)",
    "max_rounds":         "Deps.max_rounds = 2: how many times Diagnosis may ask for more evidence",
    "AZURE_*":            "workspace id, subscription, resource group, app role (azure mode); AZURE_OPENAI_* (openai reasoner)",
}
for k, v in SETTINGS.items():
    print(f"{k:<20} {v}")

TRIAGE_MODE          demo | env | azure: which connectors the agents use (here: the captured shop snapshot, as in env)
TRIAGE_REASONER      rules (default) | openai: who does the judgment (here: rules)
TRIAGE_ALERT_TOKEN   optional secret the /alerts webhook must send as ?token=
TRIAGE_VERIFY_WAIT   seconds the executor waits before checking the fix worked (here 0, nothing live to wait for)
max_rounds           Deps.max_rounds = 2: how many times Diagnosis may ask for more evidence
AZURE_*              workspace id, subscription, resource group, app role (azure mode); AZURE_OPENAI_* (openai reasoner)


## 1. Triage supervisor

`Reasoner.plan(alert) -> Plan`. **Input:** the `Alert`. **Output:** which resource is affected and which agents to run. Today every alert gets the same three evidence agents, so this step is routing, not judgment.

> **Cell reference: Triage supervisor**
>
> | | |
> |---|---|
> | **Function** | `Reasoner.plan(alert)` (`RuleReasoner.plan` in `triage/llm/rules.py`; called by the `triage` node in `agents.py`) |
> | **Purpose** | Decides which resource is affected and which evidence agents to start. |
> | **Inputs** | `Alert`. |
> | **Outputs** | `Plan` (`resource`, `agents`). Today it is always logs, metrics and change. |

In [5]:
from triage.llm.rules import RuleReasoner

reasoner = RuleReasoner()      # swap for AzureOpenAIReasoner and nothing below changes
plan = reasoner.plan(alert)
show("Plan  <- output of the Triage supervisor", plan)

--- Plan  <- output of the Triage supervisor
{
  "resource": "shop-api",
  "agents": [
    "logs",
    "metrics",
    "change"
  ]
}


## 2. The evidence agents

Each evidence agent calls **one connector** and gets back `Evidence`: a readable `summary` plus structured `detail`. The agent wraps it as a `Finding` (adds the agent name and round). In the real graph the three run **at the same time**; here they run one after another so you can read each.

The connectors read from `shop`, which serves the captured telemetry. In Azure mode the same classes' siblings (`AzureLogs`, `AzureMetrics`, `AzureChanges`) run KQL and read the Activity Log, and produce the same shape.

> **Cell reference: Metrics agent**
>
> | | |
> |---|---|
> | **Function** | `EnvMetrics.query(alert)` (`triage/connectors/env.py`, using `metrics_evidence` from `connectors/analysis.py`) |
> | **Purpose** | Reads the last minutes of request and gauge metrics and works out what changed: error rate, latency, memory, restarts, database pool. |
> | **Inputs** | `Alert`. The data comes from the shop stand-in (`shop`). |
> | **Outputs** | `Evidence`: `summary` text and `detail` numbers such as `error_rate_now` and `error_rate_before`. |

In [6]:
from triage.connectors.env import EnvChanges, EnvLogs, EnvMetrics
from triage.models import Finding

shop, shop_calls = shop_client(SCENARIO)   # shop_calls will record any write made to the "shop"
logs_connector, metrics_connector, changes_connector = EnvLogs(shop), EnvMetrics(shop), EnvChanges(shop)

# ---- Metrics agent: deps.metrics.query(alert) ----
metrics_ev = metrics_connector.query(alert)
print("METRICS summary:", metrics_ev.summary)
show("Evidence.detail from the Metrics connector", metrics_ev.detail)

METRICS summary: HTTP 5xx rate is 31% (was 0.0%).
--- Evidence.detail from the Metrics connector
{
  "error_rate_now": 0.30974,
  "error_rate_before": 0.0,
  "p95_ms_now": 104.97,
  "p95_ms_before": 107.80526315789474,
  "db_util": 0.2,
  "db_pool": 8,
  "db_in_use": 1.6,
  "restarts_last_minute": 0,
  "mem_slope_mb_per_min": 0.0,
  "mem_mb_now": 300.0
}


> **Cell reference: Logs agent (first pass) and Change agent**
>
> | | |
> |---|---|
> | **Function** | `EnvLogs.query(alert)` and `EnvChanges.recent(alert)` (`triage/connectors/env.py`) |
> | **Purpose** | Logs: groups error lines by pattern and counts them. Change: lists deploys and config changes from the last 3 days. |
> | **Inputs** | `Alert`, and for logs `deeper=False`. |
> | **Outputs** | Two `Evidence` objects. Logs `detail` has `total` and `signatures`. Change `detail` has `changes` with version and age. |

In [7]:
# ---- Logs agent, first pass: deps.logs.query(alert) ----
logs_ev = logs_connector.query(alert)               # deeper=False
print("LOGS summary:", logs_ev.summary)
show("Evidence.detail from the Logs connector (first pass)", logs_ev.detail)

# ---- Change agent: deps.changes.recent(alert) ----
change_ev = changes_connector.recent(alert)
print("\nCHANGE summary:", change_ev.summary)
show("Evidence.detail from the Change connector", change_ev.detail)

LOGS summary: 280 errors in the last 30 minutes. Most common (280x): AttributeError in PricingCalculator.apply_discount. First seen 40 seconds ago.
--- Evidence.detail from the Logs connector (first pass)
{
  "total": 280,
  "signatures": [
    {
      "sig": "AttributeError in PricingCalculator.apply_discount: '\u2026' object has no attribute '\u2026'",
      "msg": "AttributeError in PricingCalculator.apply_discount: 'NoneType' object has no attribute 'discount'",
      "count": 280,
      "first_age_s": 39.80085754394531,
      "last_age_s": 0.04649066925048828
    }
  ],
  "deeper": false
}

CHANGE summary: Deploy v482: new coupon discount rules in the pricing module (40 seconds ago).
--- Evidence.detail from the Change connector
{
  "changes": [
    {
      "kind": "deploy",
      "age_min": 0.6666223605473837,
      "detail": "Deploy v482: new coupon discount rules in the pricing module",
      "meta": {
        "version": "v482",
        "from_version": "v481"
      }
    }
  ]


> **Cell reference: Findings handed to Diagnosis**
>
> | | |
> |---|---|
> | **Function** | `Finding(...)` (`triage/models.py`; built by `gather()` in `agents.py`) |
> | **Purpose** | Wraps each `Evidence` with the agent name and round, the shape Diagnosis receives. |
> | **Inputs** | The three `Evidence` objects from the cells above. |
> | **Outputs** | `findings`, a dict of `metrics`, `logs` and `change`, each a `Finding`. |

In [8]:
# What each agent hands to Diagnosis: a Finding (this is what gather() in agents.py builds)
findings = {
    "metrics": Finding(agent="metrics", summary=metrics_ev.summary, round=1, detail=metrics_ev.detail),
    "logs":    Finding(agent="logs",    summary=logs_ev.summary,    round=1, detail=logs_ev.detail),
    "change":  Finding(agent="change",  summary=change_ev.summary,  round=1, detail=change_ev.detail),
}
for name, f in findings.items():
    print(f"{name:<8} round {f.round}: {f.summary}")

metrics  round 1: HTTP 5xx rate is 31% (was 0.0%).
logs     round 1: 280 errors in the last 30 minutes. Most common (280x): AttributeError in PricingCalculator.apply_discount. First seen 40 seconds ago.
change   round 1: Deploy v482: new coupon discount rules in the pricing module (40 seconds ago).


## 3. Diagnosis agent, first pass

`Reasoner.diagnose(alert, findings, round) -> Diagnosis`. **Input:** the alert and all three findings. **Output:** a cause, a confidence, a `kind`, and `needs_more`. Watch `confidence` and `needs_more`: with only the error summary and no stack trace, the reasoner is not sure enough to act (threshold `ENOUGH` = 0.8).

> **Cell reference: Diagnosis agent (first pass)**
>
> | | |
> |---|---|
> | **Function** | `Reasoner.diagnose(alert, findings, round)` (`RuleReasoner.diagnose`; called by the `diagnosis` node) |
> | **Purpose** | Weighs the three findings, names a cause, scores how sure it is, and says whether it needs more evidence. |
> | **Inputs** | `Alert`, `findings`, `round=1`. |
> | **Outputs** | `Diagnosis`: `cause`, `confidence`, `needs_more`, `reason`, `kind`, `params`. |

In [9]:
from triage.llm.rules import ENOUGH

diagnosis = reasoner.diagnose(alert, findings, round=1)
show("Diagnosis  <- round 1", diagnosis)
print(f"\nconfidence {diagnosis.confidence} vs threshold {ENOUGH}  ->  needs_more={diagnosis.needs_more}")
print("reason:", diagnosis.reason or "(none, it is sure)")

--- Diagnosis  <- round 1
{
  "cause": "deploy v482 introduced AttributeError in PricingCalculator.apply_discount",
  "confidence": 0.7,
  "needs_more": true,
  "reason": "The errors start right after a deploy, but the log lines do not yet say which code changed.",
  "kind": "bad-deploy",
  "params": {
    "bad_version": "v482",
    "rollback_to": "v481"
  }
}

confidence 0.7 vs threshold 0.8  ->  needs_more=True
reason: The errors start right after a deploy, but the log lines do not yet say which code changed.


## 4. The loop: Logs agent, second pass, then Diagnosis again

If `needs_more` is true and rounds remain (`max_rounds`, default 2), the graph routes to the Logs agent again with `deeper=True`: full stack traces and complete timeouts. Its new Finding **replaces** the first Logs finding. Memory-leak is confident on the first pass, so with that scenario this cell shows no change.

> **Cell reference: Logs agent (second pass) and Diagnosis again**
>
> | | |
> |---|---|
> | **Function** | `EnvLogs.query(alert, deeper=True)`, then `Reasoner.diagnose(..., round=2)` (the `logs_deeper` and `diagnosis` nodes) |
> | **Purpose** | Runs only when Diagnosis was unsure and rounds remain. Pulls stack traces, replaces the Logs finding, and diagnoses again. |
> | **Inputs** | `Alert`, `deeper=True`, the updated `findings`, `round=2`. |
> | **Outputs** | A new `Finding` for logs (adds `frames` and `pool_saturated`) and a new `Diagnosis` with higher confidence. Prints a message instead if the first pass was confident. |

In [10]:
max_rounds = 2
if diagnosis.needs_more and 1 < max_rounds:
    deeper_ev = logs_connector.query(alert, deeper=True)
    print("LOGS (second pass) summary:", deeper_ev.summary)
    findings["logs"] = Finding(agent="logs", summary=deeper_ev.summary, round=2, detail=deeper_ev.detail)
    print("new in detail:", {k: deeper_ev.detail[k] for k in ("frames", "pool_saturated") if k in deeper_ev.detail})
    diagnosis = reasoner.diagnose(alert, findings, round=2)
    show("Diagnosis  <- round 2", diagnosis)
    print(f"\nconfidence {diagnosis.confidence} vs threshold {ENOUGH}  ->  needs_more={diagnosis.needs_more}")
else:
    print("Confident on the first pass: no second Logs pass needed.")

LOGS (second pass) summary: 280 errors in the last 30 minutes. Most common (280x): AttributeError in PricingCalculator.apply_discount. First seen 40 seconds ago. The stack trace ends in pricing.py:apply_discount in all 280 of these errors.
new in detail: {'frames': [('pricing.py:apply_discount', 280)]}
--- Diagnosis  <- round 2
{
  "cause": "deploy v482 introduced AttributeError in PricingCalculator.apply_discount",
  "confidence": 0.85,
  "needs_more": false,
  "reason": "",
  "kind": "bad-deploy",
  "params": {
    "bad_version": "v482",
    "rollback_to": "v481"
  }
}

confidence 0.85 vs threshold 0.8  ->  needs_more=False


## 5. Remediation agent

`Reasoner.propose_fix(alert, diagnosis) -> FixProposal`. **Input:** the diagnosis (its `kind` and `params`). **Output:** one fix chosen from a **fixed catalog** (rollback / restart / set pool size / escalate). Even with the Azure OpenAI reasoner, the model never invents an action: this step is a lookup. `kind` and `params` are what the executor will act on; `action` and `risk` are what the person reads.

> **Cell reference: Remediation agent**
>
> | | |
> |---|---|
> | **Function** | `Reasoner.propose_fix(alert, diagnosis)` (`RuleReasoner.propose_fix`; called by the `remediation` node) |
> | **Purpose** | Turns the diagnosis into one fix from a fixed catalog: rollback, restart, set pool size, or escalate. |
> | **Inputs** | `Alert` and `Diagnosis` (mainly its `kind` and `params`). |
> | **Outputs** | `FixProposal`: `action` and `risk` for the person, `kind` and `params` for the executor. |

In [11]:
proposal = reasoner.propose_fix(alert, diagnosis)
show("FixProposal  <- shown at the approval gate", proposal)

--- FixProposal  <- shown at the approval gate
{
  "action": "roll shop-api back to v481",
  "risk": "low, v481 was running cleanly before v482",
  "kind": "rollback",
  "params": {
    "to": "v481"
  }
}


## 6. Approval gate

Not an agent: a pause. In the real graph this is LangGraph's `interrupt({"proposal": ...})`. The run's thread ends, the state is checkpointed, and `POST /runs/{id}/approval` resumes it with `"approve"` or `"reject"`. Nothing after this point can run without that decision. Here we simulate the decision; the last cell of this notebook runs the real graph so you can see the actual pause.

> **Cell reference: Approval gate**
>
> | | |
> |---|---|
> | **Function** | `interrupt({...})` in the `gate` node of `agents.py` (simulated here by the `decision` variable) |
> | **Purpose** | Pauses the run until a person approves or rejects. Nothing after this runs without that decision. |
> | **Inputs** | The `FixProposal` shown to the person and their decision, `"approve"` or `"reject"`. |
> | **Outputs** | `decision` and the name of the next step: Action executor if approved, Postmortem agent if rejected. |

In [12]:
gate_payload = {"proposal": proposal.model_dump()}   # what the person is shown
show("interrupt() payload", gate_payload)

decision = "approve"      # try "reject" and run the next cells: the executor is skipped
print("\ndecision:", decision)
print("next step:", "Action executor" if decision == "approve" else "Postmortem agent (skips the executor)")

--- interrupt() payload
{
  "proposal": {
    "action": "roll shop-api back to v481",
    "risk": "low, v481 was running cleanly before v482",
    "kind": "rollback",
    "params": {
      "to": "v481"
    }
  }
}

decision: approve
next step: Action executor


## 7. Action executor, Action connector, the resource

`ActionRunner.apply(alert, proposal) -> ActionResult`. The executor agent passes the approved `FixProposal` to the **Action connector**, the only interface in the system that can change anything. Here `EnvActions` turns `kind="rollback"` into a write to the shop (`POST /actions/rollback?to=v481`), then checks that error rate and latency came back down. The write is recorded in `shop_calls` so you can see the hop to the resource.

Because the shop here is a frozen snapshot, it cannot actually heal, so the recovery check is skipped (`verify_wait=0`). In the live Docker stack the same code waits and measures the real shop.

> **Cell reference: Action executor and Action connector**
>
> | | |
> |---|---|
> | **Function** | `ActionRunner.apply(alert, proposal)` (`EnvActions.apply` in `triage/connectors/env.py`; called by the `executor` node) |
> | **Purpose** | Applies the approved fix to the resource and checks whether the system recovered. This is the only step that changes anything. |
> | **Inputs** | `Alert` and the approved `FixProposal`. |
> | **Outputs** | `ActionResult`: `ok` (it worked), `applied` (something was done), `message`. Also the list of writes that reached the shop. |

In [13]:
from triage.connectors.env import EnvActions

action = None
if decision == "approve":
    action = EnvActions(shop, verify_wait=0).apply(alert, proposal)
    show("ActionResult  <- executor output", action)
    print("\nwrites that reached the resource:", shop_calls)
else:
    print("Rejected: nothing was applied. shop_calls =", shop_calls)

--- ActionResult  <- executor output
{
  "ok": true,
  "message": "Rolled back to v481.",
  "applied": true
}

writes that reached the resource: [('/actions/rollback', {'to': 'v481'})]


> **Cell reference: Action connector on Azure (not yet built)**
>
> | | |
> |---|---|
> | **Function** | `AzureActionsNotWired.apply(alert, proposal)` (`triage/connectors/azure.py`) |
> | **Purpose** | Shows what happens today in Azure mode: nothing is changed and the fix is handed to a person. |
> | **Inputs** | `Alert` and `FixProposal`. |
> | **Outputs** | `ActionResult` with `ok=False` and `applied=False`. |

In [14]:
# In Azure mode there is no Action connector yet. The same call is honest about it:
from triage.connectors.azure import AzureActionsNotWired  # needs: pip install -e "backend[azure]"

show("ActionResult from the Azure connector today", AzureActionsNotWired().apply(alert, proposal))

--- ActionResult from the Azure connector today
{
  "ok": false,
  "message": "Azure actions are not wired yet. Handing over to the on-call engineer.",
  "applied": false
}


## 8. Postmortem agent (always runs)

The outcome is worked out from what happened: `fixed` (applied and recovered), `unhealthy` (applied, still broken), or `rejected` (rejected, or nothing could be applied). The reasoner drafts the postmortem and a work item is filed (in-memory here; Azure DevOps Boards later).

> **Cell reference: Postmortem agent and work item**
>
> | | |
> |---|---|
> | **Function** | `Reasoner.postmortem(alert, diagnosis, proposal, outcome)` and `WorkItemSink.create(alert, postmortem)` (`DemoWorkItems`) |
> | **Purpose** | Works out the outcome (fixed, unhealthy or rejected), writes the report, and files a work item. Runs on every path. |
> | **Inputs** | `Alert`, `Diagnosis`, `FixProposal`, and `outcome` computed from the approval and the `ActionResult`. |
> | **Outputs** | `Postmortem` (`title`, `summary`) and `WorkItem` (`id`, `url`). |

In [15]:
from triage.connectors.demo import DemoWorkItems

applied = decision == "approve" and (action.applied if action else True)
fixed = applied and bool(action and action.ok)
outcome = "fixed" if fixed else "unhealthy" if applied else "rejected"
print("outcome:", outcome)

postmortem = reasoner.postmortem(alert, diagnosis, proposal, outcome)
show("Postmortem", postmortem)

work_item = DemoWorkItems().create(alert, postmortem)
show("WorkItem", work_item)

outcome: fixed
--- Postmortem
{
  "title": "shop-api failed requests above 5%: deploy v482 introduced AttributeError in PricingCalculator.apply_discount",
  "summary": "Work item created: deploy v482 introduced AttributeError in PricingCalculator.apply_discount (confidence 85%). Applied: roll shop-api back to v481. Follow-up: add a test for orders that have no coupon."
}
--- WorkItem
{
  "id": "1001",
  "url": "https://dev.azure.com/demo/triage/_workitems/edit/1001"
}


## 9. How the data changed, start to finish

> **Cell reference: Summary of how the data changed**
>
> | | |
> |---|---|
> | **Function** | None (formatting only) |
> | **Purpose** | Puts the result of every stage side by side. |
> | **Inputs** | The variables created by the cells above. |
> | **Outputs** | A printed table, one line per stage. |

In [16]:
rows = [
    ("Azure payload",       "raw JSON from Azure Monitor"),
    ("Alert",               f"{alert.title!r} on {alert.resource}, severity {alert.severity}"),
    ("Plan",                f"agents {plan.agents}"),
    ("Evidence -> Finding", "; ".join(f"{n}: {len(f.summary)} chars of summary + structured detail" for n, f in findings.items())),
    ("Diagnosis",           f"{diagnosis.kind}, confidence {diagnosis.confidence}, needs_more={diagnosis.needs_more}"),
    ("FixProposal",         f"{proposal.kind} {proposal.params} ({proposal.action})"),
    ("Approval",            decision),
    ("ActionResult",        f"ok={action.ok}, applied={action.applied}" if action else "(not run)"),
    ("Postmortem",          f"{outcome}: {postmortem.title}"),
    ("WorkItem",            work_item.url),
]
for stage, value in rows:
    print(f"{stage:<22} {value}")

Azure payload          raw JSON from Azure Monitor
Alert                  'shop-api failed requests above 5%' on shop-api, severity high
Plan                   agents ['logs', 'metrics', 'change']
Evidence -> Finding    metrics: 32 chars of summary + structured detail; logs: 211 chars of summary + structured detail; change: 78 chars of summary + structured detail
Diagnosis              bad-deploy, confidence 0.85, needs_more=False
FixProposal            rollback {'to': 'v481'} (roll shop-api back to v481)
Approval               approve
ActionResult           ok=True, applied=True
Postmortem             fixed: shop-api failed requests above 5%: deploy v482 introduced AttributeError in PricingCalculator.apply_discount
WorkItem               https://dev.azure.com/demo/triage/_workitems/edit/1001


## 10. The same thing through the real graph

The cells above called the functions by hand. This cell hands the same alert and the same data to the real LangGraph, which runs the three evidence agents in parallel, loops, **pauses at the gate**, and resumes on your decision. The printed lines are the events the website animates. (Needs `langgraph`, which `backend[dev]` installs.)

> **Cell reference: The real graph**
>
> | | |
> |---|---|
> | **Function** | `build_graph(deps)` (`triage/graph.py`) driven by `RunManager.start` and `.decide` (`triage/runs.py`) |
> | **Purpose** | Runs the same alert through the actual LangGraph: parallel evidence agents, the loop, the pause at the gate, then resume on your decision. |
> | **Inputs** | `alert`, `SCENARIO`, and the approval decision passed to `runs.decide(run.id, "approve")`. |
> | **Outputs** | The event log the website animates, status changes (`awaiting_approval`, `completed`), and the writes that reached the resource. |

In [17]:
try:
    from envhelpers import env_deps_for
    from helpers import logs as run_logs, wait_for
    from triage.graph import build_graph
    from triage.runs import RunManager
except ImportError as e:
    print("Skipping: needs the backend installed with langgraph:", e)
else:
    deps, writes = env_deps_for(SCENARIO)
    runs = RunManager(build_graph(deps), 0)
    run = runs.start(alert)
    wait_for(run, "awaiting_approval")
    print("== paused at the approval gate (status:", run.status, ")\n")
    print("\n".join(run_logs(run)))
    print("\n== a person approves ==\n")
    runs.decide(run.id, "approve")
    wait_for(run, "completed")
    print("\n".join(run_logs(run)[-6:]))
    print("\nwrites that reached the resource:", writes)

== paused at the approval gate (status: awaiting_approval )

Azure Monitor: Alert fired: shop-api failed requests above 5%
Triage API: Run created. Starting the agent graph.
Triage supervisor: Affected resource: shop-api. Starting Logs, Metrics and Change agents in parallel.
Change agent: Listing deploys and config changes from the last 3 days.
Change agent: Deploy v482: new coupon discount rules in the pricing module (40 seconds ago).
Logs agent: Running templated KQL on Log Analytics for the last 30 minutes.
Logs agent: 280 errors in the last 30 minutes. Most common (280x): AttributeError in PricingCalculator.apply_discount. First seen 40 seconds ago.
Metrics agent: Reading latency, error rate, CPU and memory from Azure Monitor.
Metrics agent: HTTP 5xx rate is 31% (was 0.0%).
Diagnosis agent: Confidence 70%. The errors start right after a deploy, but the log lines do not yet say which code changed. Asking the Logs agent for a second pass.
Logs agent: Second pass: widening the window 